# UTS Intelegensi Buatan — Kasus 6: Car Origin Classification

**NIM:** 24552011132 · **Kasus:** 6 · **Dataset:** `kasus06_cars.csv` · **Target:** `Origin` (USA, Europe, Japan)

Notebook ini disiapkan untuk mengikuti ketentuan UTS dan materi dosen: Pandas/feature engineering (Hands-On Session 3), dasar klasifikasi serta evaluasi model (Hands-On Session 4), dan pola end-to-end Joblib + Streamlit dari contoh Traffic-Accident.

**Penting tentang Google Colab:** dataset disimpan pada `/content/dataset/kasus06_cars.csv`. Folder `dataset` dan file CSV akan terlihat pada panel **Files** (ikon folder di sisi kiri Colab). Jalankan sel secara berurutan.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:
%%writefile requirements.txt
numpy>=1.24
pandas>=2.0
scikit-learn==1.8.0
matplotlib>=3.7
seaborn>=0.13
joblib>=1.3
streamlit>=1.32

Writing requirements.txt


## 0. Instalasi library

In [5]:
import os

print("Direktori aktif:", os.getcwd())
print("File requirements.txt tersedia:", os.path.exists("requirements.txt"))
print("Isi direktori:", os.listdir())

Direktori aktif: /content
File requirements.txt tersedia: True
Isi direktori: ['.config', 'dataset', 'requirements.txt', 'sample_data']


In [7]:
import os
print(os.path.exists("/content/requirements.txt"))

True


In [8]:
%pip -q install -r requirements.txt

In [9]:
import sklearn
print("Scikit-learn:", sklearn.__version__)

Scikit-learn: 1.8.0


## 1. Siapkan folder dataset di panel Files

Sel ini membuat `/content/dataset`. Jika kamu sudah upload CSV melalui panel Files ke folder `dataset`, sel akan langsung menggunakannya. Jika belum, dialog upload akan muncul dan file yang dipilih akan dipindahkan ke folder tersebut.

In [ ]:
from pathlib import Path
import os, shutil, sys

BASE_DIR = Path("/content") if "google.colab" in sys.modules else Path.cwd()
DATA_DIR = BASE_DIR / "dataset"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "kasus06_cars.csv"

# Jika CSV terlanjur diunggah ke /content (root), pindahkan ke folder dataset.
root_csv = BASE_DIR / "kasus06_cars.csv"
if not DATA_PATH.exists() and root_csv.exists():
    shutil.move(str(root_csv), str(DATA_PATH))

# Fallback upload bila file belum ada di lokasi tujuan.
if not DATA_PATH.exists():
    try:
        from google.colab import files
        print("Pilih file kasus06_cars.csv. File akan disimpan ke /content/dataset/")
        uploaded = files.upload()
        for filename in uploaded:
            if filename.lower().endswith(".csv"):
                shutil.move(str(BASE_DIR / filename), str(DATA_PATH))
                break
    except ImportError:
        raise FileNotFoundError("Letakkan kasus06_cars.csv di folder dataset/.")

if not DATA_PATH.exists():
    raise FileNotFoundError("kasus06_cars.csv belum ditemukan pada dataset/.")

print("Dataset siap:", DATA_PATH)
print("Isi folder dataset:", [p.name for p in DATA_DIR.iterdir()])

## 2. Membaca dan mengaudit dataset mentah

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
RANDOM_STATE = 42

df_raw = pd.read_csv(DATA_PATH)
print("Shape:", df_raw.shape)
print("\nTipe data:")
df_raw.info()
print("\nMissing values:")
display(df_raw.isna().sum().rename("missing_count").to_frame())
print("Duplikat baris:", df_raw.duplicated().sum())
print("\nStatistik numerik:")
display(df_raw.describe().T)
print("\nDistribusi target:")
display(df_raw["Origin"].value_counts().rename_axis("Origin").to_frame("jumlah"))
print("\nNilai unik Year:", sorted(df_raw["Year"].astype(str).unique())[:5], "...")
display(df_raw.head())

### Interpretasi audit awal

- Dataset berisi 406 baris dan 9 kolom.
- `Miles_per_Gallon` memiliki 8 nilai kosong dan `Horsepower` memiliki 6 nilai kosong. Keduanya perlu diimputasi di dalam Pipeline, setelah split.
- Tidak ditemukan baris duplikat.
- Target terdiri dari USA, Japan, dan Europe dengan jumlah berbeda; oleh karena itu split menggunakan stratifikasi dan Macro F1 menjadi metrik pemilihan utama.
- Kolom `Name` dan `Year` bukan dipakai mentah: `Name` diekstraksi menjadi merek, sedangkan `Year` dikonversi menjadi tahun model.

## 3. Exploratory Data Analysis

In [ ]:
# Distribusi target
fig, ax = plt.subplots(figsize=(7, 4))
order = df_raw["Origin"].value_counts().index
sns.countplot(data=df_raw, x="Origin", order=order, ax=ax)
ax.set_title("Distribusi Negara Asal Mobil")
ax.set_xlabel("Origin")
ax.set_ylabel("Jumlah mobil")
plt.tight_layout(); plt.show()

# Distribusi fitur numerik
numeric_cols = ["Miles_per_Gallon", "Cylinders", "Displacement", "Horsepower", "Weight_in_lbs", "Acceleration"]
df_raw[numeric_cols].hist(figsize=(13, 8), bins=20)
plt.suptitle("Distribusi Fitur Numerik", y=1.02)
plt.tight_layout(); plt.show()

# Korelasi fitur numerik (hubungan antarf fitur, bukan bukti sebab-akibat)
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(df_raw[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Korelasi Antarfitur Numerik")
plt.tight_layout(); plt.show()

# Kandidat Brand dari token pertama Name, sebelum normalisasi alias
brand_candidate = df_raw["Name"].astype("string").str.strip().str.lower().str.split().str[0]
display(brand_candidate.value_counts().rename_axis("brand_candidate").to_frame("jumlah"))

# Tahun model dari tanggal YYYY-MM-DD
year_parsed = pd.to_datetime(df_raw["Year"], errors="coerce")
print("Jumlah Year gagal dikonversi:", int(year_parsed.isna().sum()))
display(year_parsed.dt.year.value_counts().sort_index().rename_axis("Model_Year").to_frame("jumlah"))

### Catatan analisis merek

Pada dataset ini ditemukan beberapa variasi ejaan/alias yang jelas, misalnya `chevy`/`chevroelt`, `maxda`, `toyouta`, `vokswagen`/`vw`, serta `mercedes`. Kita normalisasi variasi yang jelas saja. Nama yang tidak jelas asalnya tidak ditebak atau dipetakan secara spekulatif.

Brand dapat menjadi fitur karena nama/merek tersedia saat prediksi dilakukan. Namun, jika setiap merek di data hanya terkait satu kelas Origin, Brand akan menjadi proxy yang sangat kuat. Karena itu eksperimen membandingkan model dengan dan tanpa Brand serta menambahkan validasi grouped berdasarkan merek untuk mengukur skenario merek yang belum pernah dilihat.

## 4. Buat modul feature engineering (`fitur.py`)

In [10]:
%%writefile fitur.py
"""Feature engineering deterministik untuk UTS AI Kasus 6.

Transformasi ini sengaja tidak mempelajari statistik dari seluruh dataset.
Statistik imputasi, encoding, dan scaling dipelajari oleh Pipeline sklearn
pada training fold saja.
"""
from __future__ import annotations

import re
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

RAW_FEATURES = [
    "Name", "Miles_per_Gallon", "Cylinders", "Displacement",
    "Horsepower", "Weight_in_lbs", "Acceleration", "Year",
]
NUMERIC_FEATURES = [
    "Miles_per_Gallon", "Cylinders", "Displacement", "Horsepower",
    "Weight_in_lbs", "Acceleration", "Model_Year",
]
BRAND_ALIASES = {
    # Alias/ejaan yang jelas dari pemeriksaan nilai unik dataset.
    "chevy": "chevrolet",
    "chevroelt": "chevrolet",
    "maxda": "mazda",
    "toyouta": "toyota",
    "vokswagen": "volkswagen",
    "vw": "volkswagen",
    "mercedes": "mercedes-benz",
}


def extract_brand(name: object) -> str:
    """Ambil token pertama dari nama mobil dan normalisasi alias eksplisit."""
    if pd.isna(name):
        return "Unknown"
    cleaned = re.sub(r"\s+", " ", str(name).strip().lower())
    if not cleaned:
        return "Unknown"
    token = cleaned.split(" ", 1)[0]
    return BRAND_ALIASES.get(token, token)


class CarFeatureEngineer(BaseEstimator, TransformerMixin):
    """Ubah input mentah menjadi fitur numerik dan opsional Brand."""

    def __init__(self, include_brand: bool = True):
        self.include_brand = include_brand

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("CarFeatureEngineer membutuhkan pandas DataFrame.")
        missing = sorted(set(RAW_FEATURES) - set(X.columns))
        if missing:
            raise ValueError(f"Kolom input tidak lengkap: {missing}")
        self.feature_names_out_ = NUMERIC_FEATURES + (["Brand"] if self.include_brand else [])
        return self

    def transform(self, X):
        check_is_fitted(self, "feature_names_out_")
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=RAW_FEATURES)
        d = X.copy()

        # Operasi deterministik: tidak menggunakan statistik dari data test.
        year = pd.to_datetime(d["Year"], errors="coerce")
        out = d[[
            "Miles_per_Gallon", "Cylinders", "Displacement", "Horsepower",
            "Weight_in_lbs", "Acceleration",
        ]].copy()
        out["Model_Year"] = year.dt.year.astype("float64")

        if self.include_brand:
            out["Brand"] = d["Name"].map(extract_brand).astype("object")
        return out[self.feature_names_out_]

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "feature_names_out_")
        return self.feature_names_out_


Writing fitur.py


In [11]:
import fitur

print("RAW_FEATURES:", fitur.RAW_FEATURES)
print("NUMERIC_FEATURES:", fitur.NUMERIC_FEATURES)
print("Brand Toyota:", fitur.extract_brand("toyouta corolla"))
print("Feature Engineer:", fitur.CarFeatureEngineer)

RAW_FEATURES: ['Name', 'Miles_per_Gallon', 'Cylinders', 'Displacement', 'Horsepower', 'Weight_in_lbs', 'Acceleration', 'Year']
NUMERIC_FEATURES: ['Miles_per_Gallon', 'Cylinders', 'Displacement', 'Horsepower', 'Weight_in_lbs', 'Acceleration', 'Model_Year']
Brand Toyota: toyota
Feature Engineer: <class 'fitur.CarFeatureEngineer'>


## 4A. Simpan dan jalankan skrip EDA yang dapat diulang

In [12]:
%%writefile analisis_eda.py
"""Exploratory Data Analysis untuk UTS AI Kasus 6. Tidak melakukan training."""
from pathlib import Path
import json
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from fitur import extract_brand

ROOT = Path(__file__).resolve().parent
DATA = ROOT / "dataset" / "kasus06_cars.csv"
OUT = ROOT / "output"
FIG = OUT / "figures"
OUT.mkdir(exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)
df["Brand"] = df["Name"].map(extract_brand)
df["Model_Year"] = pd.to_datetime(df["Year"], errors="coerce").dt.year

# Data dictionary
rows = [
    ("Name", "object", "Nama lengkap kendaraan; diekstraksi menjadi Brand"),
    ("Miles_per_Gallon", "float", "Konsumsi bahan bakar; 8 missing values"),
    ("Cylinders", "integer", "Jumlah silinder mesin"),
    ("Displacement", "float", "Volume/displacement mesin"),
    ("Horsepower", "float", "Tenaga mesin; 6 missing values"),
    ("Weight_in_lbs", "integer", "Bobot kendaraan dalam pon"),
    ("Acceleration", "float", "Ukuran akselerasi pada dataset"),
    ("Year", "date string", "Tanggal model; diekstraksi menjadi Model_Year"),
    ("Origin", "categorical target", "USA, Europe, Japan"),
]
pd.DataFrame(rows, columns=["column", "original_type", "description_or_decision"]).to_csv(OUT / "data_dictionary.csv", index=False)

# Target distribution
fig, ax = plt.subplots(figsize=(7, 4))
df["Origin"].value_counts().reindex(["USA", "Europe", "Japan"]).plot(kind="bar", ax=ax)
ax.set(title="Distribusi Origin", xlabel="Origin", ylabel="Jumlah")
plt.tight_layout(); plt.savefig(FIG / "target_distribution.png", dpi=160); plt.close()

# Numeric distributions
num = ["Miles_per_Gallon", "Cylinders", "Displacement", "Horsepower", "Weight_in_lbs", "Acceleration"]
axes = df[num].hist(figsize=(12, 8), bins=20)
plt.suptitle("Distribusi Fitur Numerik")
plt.tight_layout(); plt.savefig(FIG / "numeric_distributions.png", dpi=160); plt.close()

# Correlation heatmap
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(df[num].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Korelasi Antarfitur Numerik")
plt.tight_layout(); plt.savefig(FIG / "correlation_heatmap.png", dpi=160); plt.close()

# Brand frequencies; relationship with target used for modeling is generated from train only in latih.py.
df["Brand"].value_counts().rename_axis("Brand").to_frame("count").to_csv(OUT / "brand_frequency.csv")
print("EDA selesai. Shape:", df.shape)
print("Missing:\n", df.isna().sum().to_string())
print("Target:\n", df.Origin.value_counts().to_string())
print("Normalized brands:", df.Brand.nunique())
print("Tahun model:", sorted(df.Model_Year.dropna().unique().tolist()))
print("Output:", OUT)


Writing analisis_eda.py


In [13]:
!python analisis_eda.py

EDA selesai. Shape: (406, 11)
Missing:
 Name                0
Miles_per_Gallon    8
Cylinders           0
Displacement        0
Horsepower          6
Weight_in_lbs       0
Acceleration        0
Year                0
Origin              0
Brand               0
Model_Year          0
Target:
 Origin
USA       254
Japan      79
Europe     73
Normalized brands: 31
Tahun model: [1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1982]
Output: /content/output


In [ ]:
from fitur import extract_brand, CarFeatureEngineer

df_brand = df_raw.copy()
df_brand["Brand"] = df_brand["Name"].map(extract_brand)
# EDA tidak menggunakan label test untuk memilih fitur/model; relasi Brand-Origin
# yang dipakai dalam evaluasi ditinjau dari training split di latih.py.
display(df_brand["Brand"].value_counts().rename_axis("Brand").to_frame("jumlah"))
print("Jumlah Brand setelah normalisasi:", df_brand["Brand"].nunique())

# Demonstrasi ekstraksi tahun dan brand; belum melakukan imputasi/scaling/encoding.
display(CarFeatureEngineer(include_brand=True).fit_transform(df_raw.head()))

## 5. Bangun training pipeline (`latih.py`)

Ketentuan anti-data-leakage dipenuhi dengan urutan: **raw train-test split terlebih dahulu**, lalu imputasi, scaling, dan encoding dipelajari di dalam Pipeline pada setiap training fold. Model dibandingkan melalui 5-fold StratifiedKFold hanya pada data train. Eksperimen dilakukan pada Logistic Regression dan Random Forest, masing-masing dengan dan tanpa Brand.

In [14]:
%%writefile latih.py
"""Training, pemilihan model via CV, evaluasi test satu kali, dan Joblib.
Jalankan dari folder proyek: python latih.py
"""
from __future__ import annotations

import json
import warnings
from pathlib import Path
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedGroupKFold,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from fitur import CarFeatureEngineer, NUMERIC_FEATURES, RAW_FEATURES, extract_brand

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "Origin"

ROOT = Path(__file__).resolve().parent
DATA_PATH = ROOT / "dataset" / "kasus06_cars.csv"
MODEL_DIR = ROOT / "models"
OUTPUT_DIR = ROOT / "output"
FIG_DIR = OUTPUT_DIR / "figures"

CLASSES = ["USA", "Europe", "Japan"]


def build_preprocessor(include_brand: bool) -> ColumnTransformer:
    """Membangun preprocessing numerik dan opsional kategorikal Brand."""
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])

    transformers = [
        ("numeric", numeric_pipe, NUMERIC_FEATURES),
    ]

    if include_brand:
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        transformers.append(("brand", categorical_pipe, ["Brand"]))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )


def build_pipeline(model, include_brand: bool) -> Pipeline:
    """Menggabungkan feature engineering, preprocessing, dan classifier."""
    return Pipeline([
        ("feature_engineering", CarFeatureEngineer(include_brand=include_brand)),
        ("preprocessing", build_preprocessor(include_brand)),
        ("classifier", model),
    ])


def audit_data(
    df: pd.DataFrame,
    duplicate_rows_before_removal: int,
    shape_before_removal: tuple[int, int],
) -> dict:
    """Membuat ringkasan audit data setelah pemeriksaan duplikasi."""
    numeric_columns = [
        "Miles_per_Gallon",
        "Cylinders",
        "Displacement",
        "Horsepower",
        "Weight_in_lbs",
        "Acceleration",
    ]

    domain_checks = {
        f"{col}_le_zero": int((df[col].dropna() <= 0).sum())
        for col in numeric_columns
    }

    iqr_candidates = {}
    for col in numeric_columns:
        values = df[col].dropna()
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        iqr_candidates[col] = int(
            ((values < lower_bound) | (values > upper_bound)).sum()
        )

    audit = {
        "shape_before_duplicate_removal": [
            int(shape_before_removal[0]),
            int(shape_before_removal[1]),
        ],
        "shape_after_duplicate_removal": [
            int(df.shape[0]),
            int(df.shape[1]),
        ],
        "duplicate_rows_before_removal": int(duplicate_rows_before_removal),
        "duplicate_rows_after_removal": int(df.duplicated().sum()),
        "dtypes": {col: str(dtype) for col, dtype in df.dtypes.items()},
        "missing_by_column": {
            col: int(value) for col, value in df.isna().sum().items()
        },
        "target_counts": {
            str(key): int(value) for key, value in df[TARGET].value_counts().items()
        },
        "target_percent": {
            str(key): float(value * 100)
            for key, value in df[TARGET].value_counts(normalize=True).items()
        },
        "domain_checks_nonpositive_numeric": domain_checks,
        "iqr_outlier_candidates_not_automatically_removed": iqr_candidates,
        "empty_name_rows": int(
            (df["Name"].fillna("").astype(str).str.strip() == "").sum()
        ),
        "unparseable_year_rows": int(
            pd.to_datetime(df["Year"], errors="coerce").isna().sum()
        ),
        "numeric_summary": df.select_dtypes(include=np.number).describe().to_dict(),
    }

    return audit


def run_training() -> dict:
    """Menjalankan audit, CV, evaluasi final, dan penyimpanan model."""
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"Dataset tidak ditemukan: {DATA_PATH}\n"
            "Pastikan kasus06_cars.csv berada di folder dataset/."
        )

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FIG_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(DATA_PATH)

    expected = set(RAW_FEATURES + [TARGET])
    missing_cols = sorted(expected - set(df.columns))
    if missing_cols:
        raise ValueError(f"Kolom dataset tidak sesuai: {missing_cols}")

    if df[TARGET].isna().any():
        raise ValueError(
            "Target Origin memiliki nilai kosong; periksa dataset sebelum training."
        )

    # Catat kondisi duplikasi sebelum penghapusan.
    shape_before_removal = df.shape
    duplicate_count = int(df.duplicated().sum())

    # Penghapusan dilakukan secara eksplisit dan dicatat di audit.
    if duplicate_count > 0:
        df = df.drop_duplicates().reset_index(drop=True)

    X = df[RAW_FEATURES].copy()
    y = df[TARGET].astype(str).copy()

    if set(y.unique()) != set(CLASSES):
        raise ValueError(
            f"Kelas target tidak sesuai. Ditemukan: {sorted(y.unique())}"
        )

    audit = audit_data(
        df,
        duplicate_rows_before_removal=duplicate_count,
        shape_before_removal=shape_before_removal,
    )

    # Split data mentah sebelum imputasi, encoding, maupun scaling.
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    # Analisis hubungan Brand-Origin hanya menggunakan data training.
    train_brand = X_train["Name"].map(extract_brand)
    brand_origin_table = pd.crosstab(train_brand, y_train)
    brand_origin_table.to_csv(OUTPUT_DIR / "brand_origin_cross_tab.csv")

    brands_per_origin_count = int(
        (brand_origin_table.gt(0).sum(axis=1) > 1).sum()
    )

    brand_summary = {
        "normalized_unique_brands_in_train": int(train_brand.nunique()),
        "train_brands_associated_with_multiple_origins": brands_per_origin_count,
        "each_observed_train_brand_has_one_origin_label": (
            brands_per_origin_count == 0
        ),
        "alias_rules": {
            "chevy/chevroelt": "chevrolet",
            "maxda": "mazda",
            "toyouta": "toyota",
            "vokswagen/vw": "volkswagen",
            "mercedes": "mercedes-benz",
        },
    }

    audit["brand_analysis_train_only"] = brand_summary

    with open(OUTPUT_DIR / "audit_data.json", "w", encoding="utf-8") as file:
        json.dump(audit, file, indent=2, ensure_ascii=False)

    # Pemilihan model menggunakan 5-fold CV pada training set.
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    experiments = []
    best_search = None
    best_key = None
    best_score = -np.inf

    # Ablation study: membandingkan model dengan dan tanpa fitur Brand.
    for include_brand in (False, True):
        for model_name in ("Logistic Regression", "Random Forest"):
            if model_name == "Logistic Regression":
                estimator = LogisticRegression(
                    max_iter=4000,
                    solver="lbfgs",
                    random_state=RANDOM_STATE,
                )
                param_grid = {
                    "classifier__C": [0.1, 1.0, 10.0],
                    "classifier__class_weight": [None, "balanced"],
                }
            else:
                estimator = RandomForestClassifier(
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                )
                param_grid = {
                    "classifier__n_estimators": [120],
                    "classifier__max_features": ["sqrt"],
                    "classifier__min_samples_leaf": [1, 2],
                    "classifier__class_weight": [None, "balanced_subsample"],
                }

            pipeline = build_pipeline(estimator, include_brand)

            search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                scoring="f1_macro",
                cv=cv,
                n_jobs=1,
                refit=True,
                return_train_score=True,
                error_score="raise",
            )

            search.fit(X_train, y_train)

            row = {
                "model": model_name,
                "brand_feature": include_brand,
                "cv_f1_macro_mean": float(search.best_score_),
                "cv_f1_macro_std": float(
                    search.cv_results_["std_test_score"][search.best_index_]
                ),
                "cv_train_f1_macro_mean": float(
                    search.cv_results_["mean_train_score"][search.best_index_]
                ),
                "best_params": json.dumps(
                    search.best_params_,
                    ensure_ascii=False,
                    default=str,
                ),
            }
            experiments.append(row)

            key = (model_name, include_brand)
            if search.best_score_ > best_score:
                best_score = float(search.best_score_)
                best_search = search
                best_key = key

    if best_search is None:
        raise RuntimeError("Tidak ada model yang berhasil dilatih.")

    # Diagnostik tambahan untuk merek yang tidak muncul pada fold training.
    # Ini merupakan diagnostik generalisasi, bukan pengganti CV utama.
    brand_groups = X_train["Name"].map(extract_brand)

    group_cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    def group_macro_f1_scorer(estimator, X, y):
        predictions = estimator.predict(X)

        return f1_score(
            y,
            predictions,
            labels=CLASSES,
            average="macro",
            zero_division=0,
        )

    group_scores = cross_val_score(
        best_search.best_estimator_,
        X_train,
        y_train,
        cv=group_cv,
        groups=brand_groups,
        scoring=group_macro_f1_scorer,
        n_jobs=1,
    )


    group_cv_diagnostic = {
        "description": (
            "StratifiedGroupKFold grouped by normalized car brand; "
            "train split only"
        ),
        "macro_f1_mean": float(group_scores.mean()),
        "macro_f1_std": float(group_scores.std()),
        "fold_scores": [float(value) for value in group_scores],
    }

    # Baseline Dummy juga dinilai melalui CV pada training set.
    dummy_pipe = build_pipeline(
        DummyClassifier(strategy="prior"),
        include_brand=False,
    )

    dummy_scores = cross_val_score(
        dummy_pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="f1_macro",
        n_jobs=1,
    )

    baseline_cv = {
        "model": "DummyClassifier (prior)",
        "cv_f1_macro_mean": float(dummy_scores.mean()),
        "cv_f1_macro_std": float(dummy_scores.std()),
    }

    comparison = pd.DataFrame(experiments).sort_values(
        "cv_f1_macro_mean",
        ascending=False,
    )
    comparison.to_csv(OUTPUT_DIR / "perbandingan_cv.csv", index=False)

    # GridSearchCV dengan refit=True sudah melatih model terbaik pada X_train.
    final_model = best_search.best_estimator_

    # Evaluasi held-out test dilakukan satu kali setelah pemilihan model.
    y_pred = final_model.predict(X_test)
    y_proba = final_model.predict_proba(X_test)

    model_classes = list(final_model.named_steps["classifier"].classes_)

    cm = confusion_matrix(y_test, y_pred, labels=CLASSES)
    report_dict = classification_report(
        y_test,
        y_pred,
        labels=CLASSES,
        output_dict=True,
        zero_division=0,
    )

    metrics = {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
        "precision_macro": float(
            precision_score(y_test, y_pred, average="macro", zero_division=0)
        ),
        "recall_macro": float(
            recall_score(y_test, y_pred, average="macro", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y_test, y_pred, average="macro", zero_division=0)
        ),
        "f1_weighted": float(
            f1_score(y_test, y_pred, average="weighted", zero_division=0)
        ),
        "roc_auc_ovr_macro": float(
            roc_auc_score(
                y_test,
                y_proba,
                labels=model_classes,
                multi_class="ovr",
                average="macro",
            )
        ),
    }

    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual_{class_name}" for class_name in CLASSES],
        columns=[f"Predicted_{class_name}" for class_name in CLASSES],
    )

    cm_df.to_csv(OUTPUT_DIR / "confusion_matrix.csv")
    pd.DataFrame(report_dict).transpose().to_csv(
        OUTPUT_DIR / "classification_report.csv"
    )

    # Visualisasi confusion matrix.
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("Confusion Matrix - Car Origin Classification")
    plt.xlabel("Predicted label")
    plt.ylabel("Actual label")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "confusion_matrix.png", dpi=160)
    plt.close()

    # Visualisasi perbandingan eksperimen CV.
    cv_plot = comparison.copy()
    cv_plot["experiment"] = (
        cv_plot["model"]
        + cv_plot["brand_feature"].map({
            True: " + Brand",
            False: " tanpa Brand",
        })
    )

    plt.figure(figsize=(9, 5))
    sns.barplot(
        data=cv_plot,
        x="cv_f1_macro_mean",
        y="experiment",
        color="#4472C4",
    )
    plt.xlim(0, 1)
    plt.xlabel("Mean Macro F1 (5-fold CV, training set)")
    plt.ylabel("Experiment")
    plt.title("Perbandingan Model dan Ablation Brand")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "perbandingan_cv.png", dpi=160)
    plt.close()

    metadata = {
        "project": (
            "UTS Intelegensi Buatan - Kasus 6: "
            "Car Origin Classification"
        ),
        "nim": "24552011132",
        "case": 6,
        "target": TARGET,
        "classes": CLASSES,
        "probability_class_order": model_classes,
        "selected_model": best_key[0],
        "brand_feature_used": bool(best_key[1]),
        "brand_analysis": brand_summary,
        "best_cv_macro_f1": float(best_score),
        "best_params": best_search.best_params_,
        "baseline_cv_macro_f1": baseline_cv,
        "unseen_brand_group_cv_diagnostic": group_cv_diagnostic,
        "test_metrics": metrics,
        "test_confusion_matrix": cm.tolist(),
        "train_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "cv": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
        "feature_columns_raw": RAW_FEATURES,
        "available_model_years": sorted(
            int(value)
            for value in pd.to_datetime(
                df["Year"], errors="coerce"
            ).dt.year.dropna().unique()
        ),
        "sklearn_version": sklearn.__version__,
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "notes": [
            "Preprocessing statistics are learned inside Pipeline and CV training folds only.",
            "Brand alias normalization is deterministic and based on inspected dataset variants.",
            "The held-out test set is evaluated once after model selection.",
            "Group CV is a supplementary diagnostic for unseen-brand generalization.",
        ],
    }

    with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=2, ensure_ascii=False, default=str)

    with open(MODEL_DIR / "model_metadata.json", "w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=2, ensure_ascii=False, default=str)

    bundle = {
        "pipeline": final_model,
        "metadata": metadata,
        "class_order": model_classes,
        "raw_feature_columns": RAW_FEATURES,
    }

    joblib.dump(
        bundle,
        MODEL_DIR / "car_origin_pipeline.joblib",
        compress=3,
    )

    print("\n=== AUDIT DATA ===")
    print("Shape before duplicate removal:", shape_before_removal)
    print("Shape after duplicate removal:", df.shape)
    print("Missing:\n", df.isna().sum().to_string())
    print("Duplicate rows before removal:", duplicate_count)
    print("Duplicate rows after removal:", int(df.duplicated().sum()))
    print("Target:\n", y.value_counts().to_string())

    print("\n=== CV COMPARISON (TRAIN ONLY) ===")
    print(
        comparison[
            [
                "model",
                "brand_feature",
                "cv_f1_macro_mean",
                "cv_f1_macro_std",
            ]
        ].to_string(index=False)
    )

    print(
        "\nBaseline Dummy CV Macro F1:",
        round(baseline_cv["cv_f1_macro_mean"], 4),
    )
    print(
        "Unseen-brand Group CV Macro F1 (diagnostic):",
        round(group_cv_diagnostic["macro_f1_mean"], 4),
    )
    print(
        "\nSelected:",
        best_key,
        "| CV Macro F1:",
        round(best_score, 4),
    )

    print("\n=== FINAL TEST METRICS (EVALUATED ONCE) ===")
    for metric_name, value in metrics.items():
        print(f"{metric_name:22s}: {value:.4f}")

    print("\n=== CLASSIFICATION REPORT ===")
    print(
        classification_report(
            y_test,
            y_pred,
            labels=CLASSES,
            digits=4,
            zero_division=0,
        )
    )

    print("=== CONFUSION MATRIX ===")
    print(cm_df)

    print("\nModel saved:", MODEL_DIR / "car_origin_pipeline.joblib")
    print("Artifacts saved:", OUTPUT_DIR)

    return {
        "metadata": metadata,
        "comparison": comparison,
        "confusion_matrix": cm_df,
    }


if __name__ == "__main__":
    run_training()


Writing latih.py


In [17]:
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.metrics import f1_score

from fitur import extract_brand

# Memuat pipeline hasil training sebelumnya
bundle = joblib.load("models/car_origin_pipeline.joblib")
pipeline = bundle["pipeline"]

# Membaca dataset dan menggunakan pembagian train yang sama
df = pd.read_csv("dataset/kasus06_cars.csv")

from sklearn.model_selection import train_test_split

X = df[bundle["raw_feature_columns"]].copy()
y = df["Origin"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# Membuat grup berdasarkan Brand
brand_groups = X_train["Name"].map(extract_brand)

group_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

def group_macro_f1_scorer(estimator, X, y):
    predictions = estimator.predict(X)
    return f1_score(
        y,
        predictions,
        labels=["USA", "Europe", "Japan"],
        average="macro",
        zero_division=0,
    )

group_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=group_cv,
    groups=brand_groups,
    scoring=group_macro_f1_scorer,
    n_jobs=1,
)

print("Group CV fold scores:", group_scores)
print("Group CV Macro F1 Mean:", round(group_scores.mean(), 4))
print("Group CV Macro F1 Std:", round(group_scores.std(), 4))

Group CV fold scores: [0.63251029 0.6672105  0.60702184 0.59374607 0.50961538]
Group CV Macro F1 Mean: 0.602
Group CV Macro F1 Std: 0.0526


## 6. Training, pemilihan model, dan evaluasi final

In [16]:
!python latih.py

/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
        cached_call, estimator, *args, **routed_params.get(name).score
    )
  File "/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
        estimator,
    ...<2 lines>...
        pos_label=pos_label,
    )
  File "/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_scorer.py", line 96, in _cached_call
    result, _ = _get_response_values(
                ~~~~~~~~~~~~~~~~~~~~^
        estimator, *args, response_method=response_method, **kwargs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/

### Membaca hasil training

- Tabel CV digunakan untuk memilih model; skor CV berasal dari training set saja.
- `unseen_brand_group_cv_diagnostic` adalah pemeriksaan tambahan pada training set dengan kelompok merek. Ini menguji kemampuan pada merek yang tidak muncul di fold training.
- Test set terpisah hanya digunakan satu kali untuk evaluasi final setelah model dipilih.
- Perhatikan perbedaan skor random split dan grouped CV. Jangan menyimpulkan generalisasi ke merek baru hanya dari skor test random split.

In [ ]:
import json
import joblib
import pandas as pd
from IPython.display import display

with open("output/metrics.json", encoding="utf-8") as f:
    metadata = json.load(f)
print("Model terpilih:", metadata["selected_model"])
print("Brand digunakan:", metadata["brand_feature_used"])
print("CV Macro F1:", metadata["best_cv_macro_f1"])
print("Unseen-brand Group CV:", metadata["unseen_brand_group_cv_diagnostic"])
print("Metrik test:", metadata["test_metrics"])
display(pd.read_csv("output/perbandingan_cv.csv"))
display(pd.read_csv("output/confusion_matrix.csv", index_col=0))

bundle = joblib.load("models/car_origin_pipeline.joblib")
contoh_mobil = pd.DataFrame([{
    "Name": "toyota corolla", "Miles_per_Gallon": 30.0, "Cylinders": 4,
    "Displacement": 120.0, "Horsepower": 85.0, "Weight_in_lbs": 2200,
    "Acceleration": 16.0, "Year": "1980-01-01",
}])[bundle["raw_feature_columns"]]
prob = bundle["pipeline"].predict_proba(contoh_mobil)[0]
print("Contoh input baru - prediksi:", bundle["pipeline"].predict(contoh_mobil)[0])
display(pd.DataFrame({"Origin": bundle["class_order"], "Probability": prob}).sort_values("Probability", ascending=False))

## 7. Buat aplikasi Streamlit (`app.py`)

In [27]:

%%writefile app.py
"""Streamlit app: prediksi negara asal mobil dari spesifikasi mentah."""

from pathlib import Path
import json
import joblib
import pandas as pd
import streamlit as st

ROOT = Path(__file__).resolve().parent
MODEL_PATH = ROOT / "models" / "car_origin_pipeline.joblib"
METADATA_PATH = ROOT / "models" / "model_metadata.json"

st.set_page_config(
    page_title="Car Origin Predictor",
    page_icon="🚗",
    layout="wide"
)

st.title("🚗 Car Origin Predictor")
st.caption(
    "Klasifikasi negara asal mobil berdasarkan spesifikasi "
    "dan informasi model kendaraan."
)


@st.cache_resource
def load_bundle():
    if not MODEL_PATH.exists():
        return None

    loaded_bundle = joblib.load(MODEL_PATH)

    required_keys = {
        "pipeline",
        "metadata",
        "class_order",
        "raw_feature_columns",
    }

    if not isinstance(loaded_bundle, dict):
        raise ValueError("Format model bundle tidak valid.")

    missing_keys = required_keys - set(loaded_bundle.keys())

    if missing_keys:
        raise ValueError(
            f"Komponen model bundle tidak lengkap: {sorted(missing_keys)}"
        )

    if not isinstance(loaded_bundle["metadata"], dict):
        raise ValueError("Metadata model tidak valid.")

    # Gunakan metadata JSON terbaru jika tersedia.
    if METADATA_PATH.exists():
        with METADATA_PATH.open("r", encoding="utf-8") as f:
            latest_metadata = json.load(f)

        loaded_bundle["metadata"] = latest_metadata

    return loaded_bundle


bundle = load_bundle()

if bundle is None:
    st.error("Model belum tersedia. Periksa folder models.")
    st.stop()

meta = bundle["metadata"]
pipeline = bundle["pipeline"]

with st.sidebar:
    st.header("Informasi Model")
    st.write(f"**Algoritma:** {meta['selected_model']}")
    st.write(
        f"**Brand feature:** "
        f"{'Digunakan' if meta['brand_feature_used'] else 'Tidak digunakan'}"
    )
    st.write(f"**Kelas:** {', '.join(meta['classes'])}")
    st.caption(
        "Hasil prediksi merupakan estimasi model berdasarkan pola "
        "pada dataset UTS, bukan bukti asal manufaktur yang pasti."
    )


pred_tab, eval_tab, about_tab = st.tabs(
    ["Prediksi", "Evaluasi Model", "Tentang"]
)


# ==========================================================
# TAB 1: PREDIKSI
# ==========================================================

with pred_tab:
    st.subheader("Masukkan spesifikasi mobil")

    with st.form("car_prediction_form"):
        left, right = st.columns(2)

        with left:
            name = st.text_input(
                "Nama / merek dan model mobil",
                value="toyota corolla",
                help="Contoh: toyota corolla atau ford maverick"
            )

            year_options = meta.get(
                "available_model_years",
                list(range(1970, 1983))
            )

            year = st.selectbox(
                "Tahun model",
                year_options,
                index=0
            )

            mpg_unknown = st.checkbox(
                "Miles per gallon tidak diketahui",
                value=False
            )

            mpg = st.number_input(
                "Miles per gallon",
                min_value=0.0,
                max_value=100.0,
                value=23.0,
                step=0.1,
                disabled=mpg_unknown
            )

            cylinders = st.selectbox(
                "Jumlah silinder",
                [3, 4, 5, 6, 8],
                index=1
            )

        with right:
            displacement = st.number_input(
                "Displacement",
                min_value=1.0,
                max_value=600.0,
                value=120.0,
                step=1.0
            )

            hp_unknown = st.checkbox(
                "Horsepower tidak diketahui",
                value=False
            )

            horsepower = st.number_input(
                "Horsepower",
                min_value=1.0,
                max_value=500.0,
                value=90.0,
                step=1.0,
                disabled=hp_unknown
            )

            weight = st.number_input(
                "Weight (lbs)",
                min_value=500,
                max_value=8000,
                value=2500,
                step=25
            )

            acceleration = st.number_input(
                "Acceleration",
                min_value=1.0,
                max_value=40.0,
                value=15.0,
                step=0.1
            )

        submit = st.form_submit_button(
            "Prediksi negara asal",
            type="primary",
            width="stretch"
        )

    if submit:
        if not name.strip():
            st.warning(
                "Nama mobil perlu diisi agar fitur Brand bisa diekstraksi."
            )
        else:
            row = pd.DataFrame([{
                "Name": name,
                "Miles_per_Gallon": None if mpg_unknown else mpg,
                "Cylinders": cylinders,
                "Displacement": displacement,
                "Horsepower": None if hp_unknown else horsepower,
                "Weight_in_lbs": weight,
                "Acceleration": acceleration,
                "Year": f"{year}-01-01",
            }])[bundle["raw_feature_columns"]]

            pred = pipeline.predict(row)[0]
            proba = pipeline.predict_proba(row)[0]

            probs = pd.DataFrame({
                "Origin": bundle["class_order"],
                "Probability": proba
            }).sort_values(
                "Probability",
                ascending=False
            )

            st.divider()
            st.success(f"Prediksi negara asal: **{pred}**")

            c1, c2 = st.columns([1, 2])

            with c1:
                st.metric("Predicted origin", pred)
                st.metric(
                    "Probabilitas tertinggi",
                    f"{probs.iloc[0]['Probability']:.1%}"
                )

            with c2:
                st.write("**Probabilitas setiap kelas**")
                st.bar_chart(probs.set_index("Origin"))
                st.dataframe(
                    probs.style.format({"Probability": "{:.1%}"}),
                    hide_index=True,
                    width="stretch"
                )


# ==========================================================
# TAB 2: EVALUASI MODEL
# ==========================================================

with eval_tab:
    st.subheader("Evaluasi pada held-out test set")

    st.caption(
        "Metrik berikut berasal dari evaluasi final satu kali pada "
        "data test setelah pemilihan model melalui cross-validation "
        "data train."
    )

    metrics = meta["test_metrics"]
    metric_items = list(metrics.items())

    for start in range(0, len(metric_items), 3):
        cols = st.columns(3)

        for col, (key, val) in zip(
            cols,
            metric_items[start:start + 3]
        ):
            col.metric(
                key.replace("_", " ").title(),
                f"{val:.3f}"
            )

    st.divider()

    st.write("**Confusion matrix**")

    cm = pd.DataFrame(
        meta["test_confusion_matrix"],
        index=meta["classes"],
        columns=meta["classes"]
    )

    st.dataframe(cm, width="stretch")

    st.divider()

    st.write("**Perbandingan model pada cross-validation**")

    cv_path = ROOT / "output" / "perbandingan_cv.csv"

    if cv_path.exists():
        cv_results = pd.read_csv(cv_path)
        st.dataframe(cv_results, width="stretch")
    else:
        st.warning("File perbandingan_cv.csv tidak ditemukan.")

    # ------------------------------------------------------
    # GROUP CROSS-VALIDATION
    # ------------------------------------------------------

    st.divider()

    st.subheader("Evaluasi generalisasi terhadap Brand baru")

    st.caption(
        "Evaluasi tambahan menggunakan StratifiedGroupKFold. "
        "Data dikelompokkan berdasarkan Brand yang telah dinormalisasi "
        "dan hanya menggunakan data train."
    )

    group_result = meta.get(
        "unseen_brand_group_cv_diagnostic",
        {}
    )

    group_mean = group_result.get("macro_f1_mean")
    group_std = group_result.get("macro_f1_std")
    fold_scores = group_result.get("fold_scores", [])

    if group_mean is not None and group_std is not None:
        col1, col2, col3 = st.columns(3)

        col1.metric(
            "Group CV Macro F1",
            f"{group_mean:.4f}"
        )

        col2.metric(
            "Standar deviasi",
            f"{group_std:.4f}"
        )

        col3.metric(
            "Jumlah fold",
            len(fold_scores)
        )

        if fold_scores:
            fold_table = pd.DataFrame({
                "Fold": [
                    f"Fold {i + 1}"
                    for i in range(len(fold_scores))
                ],
                "Macro F1": fold_scores
            })

            st.write("**Hasil setiap fold**")

            st.dataframe(
                fold_table.style.format({
                    "Macro F1": "{:.4f}"
                }),
                hide_index=True,
                width="stretch"
            )

        st.info(
            "Interpretasi: Skor Group CV lebih rendah dibandingkan "
            "cross-validation biasa. Hal ini menunjukkan bahwa model "
            "sangat terbantu oleh fitur Brand dan kemampuan prediksinya "
            "menurun ketika menghadapi Brand yang tidak ditemukan "
            "pada data train fold terkait. Hasil ini merupakan "
            "evaluasi diagnostik, bukan pengganti evaluasi held-out test set."
        )

    else:
        st.warning(
            "Hasil Group CV belum tersedia pada metadata."
        )


# ==========================================================
# TAB 3: TENTANG
# ==========================================================

with about_tab:
    st.subheader("Tentang proyek")

    st.write(
        "Proyek UTS Intelegensi Buatan - Kasus 6. "
        "Target klasifikasi adalah Origin dengan tiga kelas: "
        "USA, Europe, dan Japan."
    )

    st.write(f"Model terpilih: **{meta['selected_model']}**")
    st.write(f"Brand digunakan: **{meta['brand_feature_used']}**")

    st.write(
        f"Data train: **{meta['train_rows']}** | "
        f"Data test: **{meta['test_rows']}**"
    )

    st.write(f"Cross-validation: **{meta['cv']}**")

    st.info(
        "Probabilitas menunjukkan tingkat keyakinan relatif model "
        "pada tiga kelas yang dipelajari. Probabilitas bukan jaminan "
        "kebenaran faktual."
    )

    st.caption(
        "Group CV merupakan evaluasi tambahan untuk mengamati "
        "generalisasi model terhadap Brand yang tidak muncul "
        "pada data pelatihan fold terkait."
    )


Overwriting app.py


## 8. Jalankan Streamlit di Google Colab

Streamlit berjalan pada port lokal Colab. Untuk membuka UI di browser, sel berikut membuat Cloudflare Quick Tunnel sementara. URL tunnel bersifat publik dan sementara; jangan masukkan data pribadi atau rahasia ke aplikasi demo ini.

In [25]:
# Jalankan server Streamlit di background
!nohup streamlit run app.py --server.headless true --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

# Pasang cloudflared untuk tunnel sementara (tidak perlu API token)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!nohup cloudflared tunnel --url http://127.0.0.1:8501 --no-autoupdate > /content/cloudflared.log 2>&1 &

import time, re
from pathlib import Path
url = None
for _ in range(30):
    log = Path("/content/cloudflared.log").read_text(errors="ignore") if Path("/content/cloudflared.log").exists() else ""
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)
    if match:
        url = match.group(0)
        break
    time.sleep(2)
if url:
    print("Buka aplikasi Streamlit:", url)
else:
    print("URL tunnel belum ditemukan. Periksa log:")
    print(Path("/content/cloudflared.log").read_text(errors="ignore")[-3000:])
    print(Path("/content/streamlit.log").read_text(errors="ignore")[-3000:])

Buka aplikasi Streamlit: https://supposed-duncan-cover-promising.trycloudflare.com


## 9. Troubleshooting dan pemeriksaan file

In [ ]:
from pathlib import Path
for folder in ["dataset", "models", "output", "output/figures"]:
    p = Path(folder)
    print(f"{folder}/", [x.name for x in p.iterdir()] if p.exists() else "belum ada")

# Jika aplikasi gagal dibuka, lihat log Streamlit dan Cloudflare.
print("\nSTREAMLIT LOG")
print(Path("/content/streamlit.log").read_text(errors="ignore")[-2500:])

## 10. Unduh hasil proyek

Sel ini mengemas dataset, kode sumber, model hasil training, output evaluasi, dan notebook menjadi satu ZIP agar mudah disimpan.

In [ ]:
import shutil
from google.colab import files

package_dir = Path("/content/UTS_AI_Kasus06_Hasil")
if package_dir.exists():
    shutil.rmtree(package_dir)
package_dir.mkdir(parents=True)
for item in ["dataset", "models", "output", "analisis_eda.py", "fitur.py", "latih.py", "app.py", "requirements.txt", "README.md"]:
    src = Path("/content") / item
    dst = package_dir / item
    if src.is_dir():
        shutil.copytree(src, dst)
    elif src.exists():
        shutil.copy2(src, dst)
zip_path = shutil.make_archive("/content/UTS_AI_Kasus06_Hasil", "zip", root_dir=package_dir)
print("ZIP dibuat:", zip_path)
files.download(zip_path)

In [1]:
import os
import shutil

os.makedirs("/content/dataset", exist_ok=True)

shutil.move(
    "/kasus06_cars.csv",
    "/content/dataset/kasus06_cars.csv"
)

print("Dataset berhasil dipindahkan!")
print("Lokasi:", os.path.exists("/content/dataset/kasus06_cars.csv"))

Dataset berhasil dipindahkan!
Lokasi: True


In [18]:

import json
from pathlib import Path

paths = [
    Path("models/model_metadata.json"),
    Path("output/metrics.json")
]

group_cv_result = {
    "description": (
        "StratifiedGroupKFold grouped by normalized car brand; "
        "train split only"
    ),
    "macro_f1_mean": 0.6020,
    "macro_f1_std": 0.0526,
    "fold_scores": [
        0.63251029,
        0.66721050,
        0.60702184,
        0.59374607,
        0.50961538
    ]
}

for path in paths:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    data["unseen_brand_group_cv_diagnostic"] = group_cv_result

    with path.open("w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            indent=2,
            ensure_ascii=False,
            allow_nan=False
        )

    print(f"Berhasil diperbarui: {path}")

print("\nSemua metadata Group CV berhasil diperbarui!")


Berhasil diperbarui: models/model_metadata.json
Berhasil diperbarui: output/metrics.json

Semua metadata Group CV berhasil diperbarui!


In [20]:
!streamlit --version

Streamlit, version 1.67.0


In [21]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

In [22]:
!sleep 3
!cat /content/streamlit.log



2026-11-21 03:21:28.326 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.229.177.75:8501



In [24]:
!npx --yes localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋^C
